# Grouped Query Attention
*Multiple query heads share one key-value head — the memory-efficient middle ground between MHA and MQA.*

---
**Component:** `learning/leetgpu/grouped_query_attention.ipynb`
**Builds on:** `naive_gemm.py`, `softmax_attention_cutedsl.py`


## Abstract

*Imagine an office where every salesperson (query head) has their own dedicated filing cabinet full of customer records (key-value cache). That's Multi-Head Attention. Grouped Query Attention says: why not share filing cabinets? Six salespeople can share one cabinet as long as they take turns looking things up. You go from 32 cabinets to 8, freeing up three-quarters of the office space — while each salesperson still has their own unique way of querying the shared records. At inference time, smaller KV caches mean less data to shuttle from memory, which directly translates to faster token generation.*

---

## What Is GQA?

During autoregressive inference, the model reads stored key and value vectors for every token it has generated so far. The more query heads you have, the more KV vectors you must store — and reading them from HBM is the bottleneck, not the arithmetic.

Standard Multi-Head Attention (MHA) gives every query head its own dedicated key and value head. If you have 32 query heads, you have 32 KV heads — and your KV cache is proportionally large. Multi-Query Attention (MQA) goes to the opposite extreme: one shared KV head for everyone. Quality suffers. Grouped Query Attention (GQA) splits the difference:

```
MHA  (G = H = 32):  Q0→K0   Q1→K1  ... Q31→K31     32 KV heads
GQA  (G = 8):       Q0→K0   Q1→K0   Q2→K0   Q3→K0  ←─ group 0
                    Q4→K1   Q5→K1   Q6→K1   Q7→K1  ←─ group 1
                    ...                              8 KV heads total
MQA  (G = 1):       Q0→K0   Q1→K0  ... Q31→K0       1 KV head
```

The routing rule is just one line: `kv_h = query_h // group_size`. Every query head in the same group reads from the same KV head, reducing the KV cache by a factor of `H/G`.

**Where you'll see it:** LLaMA-2/3, Mistral, DeepSeek — any model designed for long-context generation at scale.

## Level 0 — Pure Python, No Imports

Let's start from the very bottom: nested `for` loops, plain Python lists, no libraries. The goal is to make the `kv_h = h // group_size` routing as visible as possible before we start optimizing.

In [ ]:
import math

# ── tiny fixed inputs ──
T, d_head, H, G = 3, 2, 4, 2   # seq_len=3, head_dim=2, 4 Q-heads, 2 KV-heads
group_size = H // G              # 2 query heads share each KV head

# Q[head][position][dim]
Q = [
    [[0.6, 0.3], [0.1, 0.9], [0.4, 0.5]],   # head 0  → KV-head 0
    [[0.8, 0.2], [0.5, 0.4], [0.3, 0.7]],   # head 1  → KV-head 0
    [[0.2, 0.7], [0.6, 0.1], [0.9, 0.3]],   # head 2  → KV-head 1
    [[0.5, 0.6], [0.3, 0.8], [0.7, 0.2]],   # head 3  → KV-head 1
]
# K[kv_head][position][dim],  V[kv_head][position][dim]
K = [
    [[0.7, 0.4], [0.2, 0.8], [0.5, 0.3]],   # kv_head 0
    [[0.3, 0.6], [0.8, 0.1], [0.4, 0.7]],   # kv_head 1
]
V = [
    [[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]],
    [[0.8, 0.2], [0.3, 0.7], [0.6, 0.4]],
]

output = [[[0.0] * d_head for _ in range(T)] for _ in range(H)]

for h in range(H):
    kv_h = h // group_size          # ← the only GQA-specific line
    for t in range(T):
        # dot products
        scores = []
        for s in range(T):
            score = 0.0
            for k in range(d_head):
                score += Q[h][t][k] * K[kv_h][s][k]
            scores.append(score / math.sqrt(d_head))
        # stable softmax
        m = max(scores)
        exps = [math.exp(sc - m) for sc in scores]
        Z = sum(exps)
        weights = [e / Z for e in exps]
        # weighted value sum
        for k in range(d_head):
            output[h][t][k] = sum(weights[s] * V[kv_h][s][k] for s in range(T))

print("GQA output[head][token][dim]:")
for h in range(H):
    for t in range(T):
        print(f"  head={h}(KV={h//group_size}) token={t}: {[round(v, 4) for v in output[h][t]]}")


## Verbose Version — Tracing the Sharing

The code above computed the right answer, but you had to squint to see the sharing happen. Let's slow it down and print exactly which KV head each query head reads from at each step. This makes the group structure impossible to miss.

In [ ]:
print("=== GQA attention trace ===")
for h in range(H):
    kv_h = h // group_size
    print(f"\n[Q-head {h}] reads K/V from kv_head={kv_h}  (group_size={group_size})")
    for t in range(T):
        # scores
        scores = []
        for s in range(T):
            raw = sum(Q[h][t][k] * K[kv_h][s][k] for k in range(d_head))
            score = raw / math.sqrt(d_head)
            scores.append(score)
            print(f"  Q[{h}][t={t}] · K[{kv_h}][s={s}] / sqrt({d_head}) = {score:.4f}")
        # softmax
        m = max(scores)
        exps = [math.exp(sc - m) for sc in scores]
        Z = sum(exps)
        weights = [e / Z for e in exps]
        print(f"  attn_weights t={t}: {[round(w, 4) for w in weights]}")


## The Formula

$$\boxed{\text{GQA}(Q, K, V)_h = \text{softmax}\!\left(\frac{Q_h\, K_{\lfloor h/g \rfloor}^\top}{\sqrt{d_k}}\right) V_{\lfloor h/g \rfloor}}$$

The only difference from standard attention is the subscript on $K$ and $V$. Instead of indexing by $h$ (the query head), you index by $\lfloor h/g \rfloor$ — the group that head $h$ belongs to.

| Symbol | Meaning |
|--------|---------|
| $H$ | Total query heads |
| $G$ | KV heads ($G \leq H$, $G$ divides $H$) |
| $g = H/G$ | Queries per KV group (the group size) |
| $Q_h \in \mathbb{R}^{T \times d_k}$ | Queries for head $h$ |
| $K_{\lfloor h/g \rfloor}$ | Keys shared by the $g$ query heads in the same group |
| $V_{\lfloor h/g \rfloor}$ | Values shared by the same group |

> **Say it out loud:** "GQA output for query head $h$ is standard softmax attention, except the key and value matrices are looked up by $\lfloor h/g \rfloor$ instead of $h$ — that floor operation is literally the only change."

**Practical savings.** At $H = 32$, $G = 8$, the KV cache shrinks by $32/8 = 4\times$. For a 70B model running at batch size 1, that can mean the difference between fitting on two GPUs versus needing eight.

## CuTe Layout Python Emulation

Before going to actual GPU code, it's useful to express the same computation using explicit `Layout` and `Tensor` objects — a Python proxy for how CuTe thinks about memory. The head dimension maps to a row offset, the group mapping becomes an integer division in the stride calculation. This step bridges "what the math says" and "what the hardware sees."

In [ ]:
class Layout:
    def __init__(self, shape, stride=None):
        self.shape = tuple(shape)
        if stride is None:
            s, running = [], 1
            for d in reversed(shape):
                s.insert(0, running); running *= d
            self.stride = tuple(s)
        else:
            self.stride = tuple(stride)
    def __call__(self, *coord):
        return sum(c * s for c, s in zip(coord, self.stride))
    def __repr__(self): return f"Layout(shape={self.shape}, stride={self.stride})"

def softmax_vec(v):
    m = max(v); exps = [math.exp(x - m) for x in v]; Z = sum(exps)
    return [e / Z for e in exps]

def gqa_cute_layout(Q_flat, K_flat, V_flat, H, G, T, d):
    group_size = H // G
    layout_Q  = Layout([H, T, d])   # row-major over (head, token, dim)
    layout_KV = Layout([G, T, d])   # KV heads
    out = [0.0] * (H * T * d)
    layout_out = Layout([H, T, d])

    for h in range(H):
        kv_h = h // group_size
        for t in range(T):
            scores = []
            for s in range(T):
                raw = sum(
                    Q_flat[layout_Q(h, t, k)] * K_flat[layout_KV(kv_h, s, k)]
                    for k in range(d)
                )
                scores.append(raw / math.sqrt(d))
            weights = softmax_vec(scores)
            for dk in range(d):
                out[layout_out(h, t, dk)] = sum(
                    weights[s] * V_flat[layout_KV(kv_h, s, dk)] for s in range(T)
                )
    return out

# Flatten our Q/K/V lists into 1-D sequences
Q_flat = [Q[h][t][k] for h in range(H) for t in range(T) for k in range(d_head)]
K_flat = [K[g][t][k] for g in range(G) for t in range(T) for k in range(d_head)]
V_flat = [V[g][t][k] for g in range(G) for t in range(T) for k in range(d_head)]

out_flat = gqa_cute_layout(Q_flat, K_flat, V_flat, H, G, T, d_head)
print("CuTe Layout GQA output (head=0, all tokens):")
lo = Layout([H, T, d_head])
for t in range(T):
    vals = [round(out_flat[lo(0, t, k)], 4) for k in range(d_head)]
    print(f"  token {t}: {vals}")

# cross-check against naive output
tol = 1e-9
for h in range(H):
    for t in range(T):
        for k in range(d_head):
            assert abs(out_flat[lo(h, t, k)] - output[h][t][k]) < tol
print("✓ matches naive output")


## CuTeDSL Implementation

In the CuTeDSL kernel, each GPU thread is responsible for exactly one output element `out[h, m, d_out]`. The thread computes the full two-pass online softmax for its row, accessing `mK[kv_h, s, k]` and `mV[kv_h, s, d_out]` where `kv_h = h // group_size`. That one substitution is all that distinguishes this from the `softmax_attention_cutedsl.py` baseline.

```python
@cute.kernel
def gqa_kernel(mQ, mK, mV, mOut, scale, group_size):
    ...
    kv_h = h // group_size   # ← the only GQA-specific line
    ...
    for s in cutlass.range(N, unroll=1):
        score = score + mQ[h,m,k] * mK[kv_h,s,k]   # shared KV head
    ...
    acc = acc + p * mV[kv_h, s, d_out]              # shared values
```

> **Performance note.** Having fewer KV heads means fewer distinct cache lines to load from HBM. When $H/G = 4$, four threads share the same KV data — and with good scheduling, the L2 cache can serve three of those four requests for free after the first fetch.

## Optimization Ladder

| Version | KV-head index | Threads | Notes |
|---------|--------------|---------|-------|
| Pure Python | `h // group_size` | 1 (serial) | Readable reference |
| CuTe Layout | same | 1 (serial Python) | Verifies stride/offset math |
| `gqa_kernel` (CuTeDSL) | `kv_h = h // group_size` | 1 per output element | Direct port of `softmax_attention_cutedsl.py` |

**KV cache at inference time** — what you're actually saving:

| Config | KV heads | Cache per layer per token | vs MHA |
|--------|---------|--------------------------|--------|
| MHA $H=32$ | 32 | $2 \times 32 \times d_k$ floats | — |
| GQA $H=32, G=8$ | 8 | $2 \times 8 \times d_k$ floats | **4× smaller** |
| MQA $H=32, G=1$ | 1 | $2 \times 1 \times d_k$ floats | **32× smaller** |

A 4× KV reduction directly translates to 4× more sequences you can serve simultaneously on the same GPU, or 4× longer context at the same memory budget.